
# Naive Bayes
Domingos & Pazzani (1997): https://link.springer.com/article/10.1023/A:1007413511361




Complement Naive Bayes (2003): https://people.csail.mit.edu/jrennie/papers/icml03-nb.pdf




Tree-Augmented Naive Bayes (1997): https://link.springer.com/article/10.1023/A:1007465528199

# Multinomial Naive Bayes

In [1]:
# Arabic Sentiment Analysis — Multinomial Naive Bayes


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

# MultinomialNB requires non-negative features.
# TF-IDF with sublinear_tf=True already produces non-negative values,
# but we pass through MinMaxScaler as an extra guarantee.
ALPHA = 1.0    # Laplace smoothing parameter

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# Stack TF-IDF + emoji features (all non-negative — safe for MultinomialNB)
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Multinomial Naive Bayes
# alpha: Laplace (additive) smoothing — prevents zero probabilities for
# words that appear in test but not in a given training class.
print("=" * 60)
print("  🟢 Training: Multinomial Naive Bayes")
print("=" * 60)

clf = MultinomialNB(alpha=ALPHA)
clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Multinomial NB  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🟢 Training: Multinomial Naive Bayes

────────────────────────────────────────────────────────────
  Multinomial NB  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8214
  Precision : 0.8215
  Recall    : 0.8214
  F1-Score  : 0.8213

  Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.81      0.82       543
           1       0.81      0.83      0.82       543

    accuracy                           0.82      1086
   macro avg       0.82      0.82      0.82      1086
weighted avg       0.82      0.82      0.82      1086

  Confusion Matrix:
[[440 103]
 [ 91 452]]


────────────────────────────────────────────────────────

# Complement Naive Bayes

In [2]:
# Arabic Sentiment Analysis — Complement Naive Bayes (CNB)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

ALPHA     = 1.0    # Laplace smoothing
NORM      = True   # Apply row-normalisation (recommended in original paper)

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 3),
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

# ComplementNB requires non-negative features — TF-IDF satisfies this
X_train = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

print(f"Final feature shape (train) : {X_train.shape}\n")

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
print(f"Classes : {le.classes_}  →  {list(range(len(le.classes_)))}\n")

# 4. MODEL — Complement Naive Bayes
# norm=True applies the weight normalisation described in Section 3.2
# of the original CNB paper — further reduces the effect of long documents.
print("=" * 60)
print("  🟢 Training: Complement Naive Bayes")
print("=" * 60)

clf = ComplementNB(alpha=ALPHA, norm=NORM)
clf.fit(X_train, y_train)

# 5. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  Complement NB  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  clf.predict(X_val),  "Validation Set")
evaluate(y_test, clf.predict(X_test), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,726
Final feature shape (train) : (5067, 7728)

Classes : [0 1]  →  [0, 1]

  🟢 Training: Complement Naive Bayes

────────────────────────────────────────────────────────────
  Complement NB  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.8177
  Precision : 0.8240
  Recall    : 0.8177
  F1-Score  : 0.8168

  Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.75      0.80       543
           1       0.78      0.89      0.83       543

    accuracy                           0.82      1086
   macro avg       0.82      0.82      0.82      1086
weighted avg       0.82      0.82      0.82      1086

  Confusion Matrix:
[[406 137]
 [ 61 482]]


──────────────────────────────────────────────────────────

# Tree-Augmented Naive Bayes

In [3]:
# Arabic Sentiment Analysis — Tree-Augmented Naive Bayes (TAN)


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.sparse import hstack, csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# CONFIGURATION
RANDOM_STATE  = 42
TRAIN_PATH    = "/content/train_stemmed.csv"
VAL_PATH      = "/content/val_stemmed.csv"
TEST_PATH     = "/content/test_stemmed.csv"
TEXT_COL      = "clean_text_stemmed"
LABEL_COL     = "labels"
HAS_EMOJI_COL = "has_emoji"
EMOJI_CNT_COL = "emoji_count"

TOP_K_FEATURES = 500   # TAN is O(K²) in features — keep manageable
N_BINS         = 5     # discretise continuous TF-IDF values into bins
ALPHA          = 1.0   # Laplace smoothing for CPTs

TFIDF_PARAMS = dict(
    analyzer      = "word",
    ngram_range   = (1, 2),      # bigrams sufficient for TAN (K is small)
    max_features  = 50_000,
    sublinear_tf  = True,
    min_df        = 2,
    max_df        = 0.95,
    strip_accents = None,
    token_pattern = r"(?u)\b\w+\b",
)

# 1. LOAD DATA
print("=" * 60)
print("  📂 Loading Data")
print("=" * 60)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
print(f"Label distribution (train):\n{train_df[LABEL_COL].value_counts()}\n")

# 2. FEATURE ENGINEERING — TF-IDF + Emoji, then feature selection
print("=" * 60)
print("  ⚙️  Building Features")
print("=" * 60)

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

X_train_tfidf = tfidf.fit_transform(train_df[TEXT_COL].fillna("").astype(str))
X_val_tfidf   = tfidf.transform(val_df[TEXT_COL].fillna("").astype(str))
X_test_tfidf  = tfidf.transform(test_df[TEXT_COL].fillna("").astype(str))

print(f"TF-IDF vocab size : {len(tfidf.vocabulary_):,}")

def emoji_sparse(df):
    return csr_matrix(df[[HAS_EMOJI_COL, EMOJI_CNT_COL]].fillna(0).values.astype(float))

X_train_full = hstack([X_train_tfidf, emoji_sparse(train_df)])
X_val_full   = hstack([X_val_tfidf,   emoji_sparse(val_df)])
X_test_full  = hstack([X_test_tfidf,  emoji_sparse(test_df)])

# 3. ENCODE LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_val   = le.transform(val_df[LABEL_COL])
y_test  = le.transform(test_df[LABEL_COL])
n_classes = len(le.classes_)
print(f"Classes : {le.classes_}  →  {list(range(n_classes))}\n")

# 4. SELECT TOP-K FEATURES (chi² filter on training data only)
print(f"  🔍 Selecting top {TOP_K_FEATURES} features by χ² score …")
selector = SelectKBest(chi2, k=TOP_K_FEATURES)
X_train_sel = selector.fit_transform(X_train_full, y_train).toarray()
X_val_sel   = selector.transform(X_val_full).toarray()
X_test_sel  = selector.transform(X_test_full).toarray()
print(f"   Selected feature shape : {X_train_sel.shape}\n")

# 5. DISCRETISE FEATURES (TAN requires discrete values)
def discretise(X_train: np.ndarray, X_others: list, n_bins: int):
    """Bin features into [0, n_bins-1] using quantile boundaries from train."""
    from sklearn.preprocessing import KBinsDiscretizer
    kbd = KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile")
    X_train_d = kbd.fit_transform(X_train).astype(int)
    X_others_d = [kbd.transform(X).astype(int) for X in X_others]
    return X_train_d, X_others_d, kbd

X_train_d, (X_val_d, X_test_d), _ = discretise(
    X_train_sel, [X_val_sel, X_test_sel], N_BINS
)

# 6. TAN — Chow-Liu Tree Construction
print("=" * 60)
print("  🌳 Building TAN (Chow-Liu tree on Conditional Mutual Information)")
print("=" * 60)

K = X_train_d.shape[1]

def conditional_mutual_information(X: np.ndarray, y: np.ndarray,
                                    n_bins: int, n_classes: int,
                                    alpha: float = 1.0) -> np.ndarray:
    """
    Compute I(Xi ; Xj | C) for all feature pairs i, j.
    Returns a symmetric (K x K) CMI matrix.
    """
    n = len(y)
    cmi = np.zeros((K, K))

    # Pre-count P(Xi=a, C=c) and P(Xi=a, Xj=b, C=c)
    for i in range(K):
        for j in range(i + 1, K):
            val = 0.0
            for c in range(n_classes):
                mask_c = (y == c)
                n_c    = mask_c.sum()
                for a in range(n_bins):
                    mask_ca = mask_c & (X[:, i] == a)
                    n_ca    = mask_ca.sum()
                    if n_ca == 0:
                        continue
                    for b in range(n_bins):
                        n_cab = (mask_ca & (X[:, j] == b)).sum()
                        n_cb  = (mask_c  & (X[:, j] == b)).sum()
                        # Laplace-smoothed probabilities
                        p_cab_c = (n_cab + alpha) / (n_c   + n_bins * n_bins * alpha)
                        p_ca_c  = (n_ca  + alpha) / (n_c   + n_bins * alpha)
                        p_cb_c  = (n_cb  + alpha) / (n_c   + n_bins * alpha)
                        p_c     = n_c / n
                        val += p_c * p_cab_c * np.log(
                            p_cab_c / (p_ca_c * p_cb_c + 1e-12) + 1e-12
                        )
            cmi[i, j] = cmi[j, i] = max(val, 0.0)
    return cmi

print(f"   Computing {K}×{K} CMI matrix (may take a moment) …")
cmi_matrix = conditional_mutual_information(
    X_train_d, y_train, N_BINS, n_classes, ALPHA
)

# Maximum spanning tree on CMI = Chow-Liu tree
# scipy's minimum_spanning_tree finds minimum, so negate CMI
neg_cmi = csr_matrix(-cmi_matrix)
mst = minimum_spanning_tree(neg_cmi).toarray()
mst = (mst != 0).astype(int)   # adjacency matrix of the tree

# Root the tree at feature 0 and extract parent array
def root_tree(adj: np.ndarray, root: int = 0):
    """BFS to assign a parent to each node in the undirected tree."""
    n   = adj.shape[0]
    par = [-1] * n
    visited = [False] * n
    queue   = [root]
    visited[root] = True
    while queue:
        node = queue.pop(0)
        for nb in range(n):
            if adj[node, nb] and not visited[nb]:
                par[nb]    = node
                visited[nb] = True
                queue.append(nb)
    return par

parents = root_tree(mst + mst.T, root=0)
print(f"   Chow-Liu tree built. Root : feature 0.\n")

# 7. TAN — Learn Conditional Probability Tables (CPTs)
# P(C) — class prior
n_train  = len(y_train)
p_class  = np.array([(y_train == c).sum() for c in range(n_classes)],
                    dtype=float)
p_class  = (p_class + ALPHA) / (n_train + n_classes * ALPHA)

# P(Xi | C)         — root node, no parent feature
# P(Xi | Xj, C)     — all other nodes, parent feature Xj
def learn_cpts(X: np.ndarray, y: np.ndarray, parents: list,
               n_bins: int, n_classes: int, alpha: float):
    """Returns list of K CPT arrays."""
    n = len(y)
    cpts = []
    for i, par in enumerate(parents):
        if par == -1:
            # Shape: (n_classes, n_bins)
            cpt = np.zeros((n_classes, n_bins))
            for c in range(n_classes):
                for a in range(n_bins):
                    cpt[c, a] = ((y == c) & (X[:, i] == a)).sum()
                cpt[c] = (cpt[c] + alpha) / (cpt[c].sum() + n_bins * alpha)
        else:
            # Shape: (n_classes, n_bins_parent, n_bins)
            cpt = np.zeros((n_classes, n_bins, n_bins))
            for c in range(n_classes):
                for b in range(n_bins):  # parent value
                    for a in range(n_bins):
                        cpt[c, b, a] = (
                            (y == c) & (X[:, par] == b) & (X[:, i] == a)
                        ).sum()
                    cpt[c, b] = (cpt[c, b] + alpha) / (cpt[c, b].sum() + n_bins * alpha)
        cpts.append(cpt)
    return cpts

print("  📚 Learning CPTs from training data …")
cpts = learn_cpts(X_train_d, y_train, parents, N_BINS, n_classes, ALPHA)

# 8. TAN — Inference (log-probability classification)
def tan_predict(X: np.ndarray) -> np.ndarray:
    """Classify each sample using the TAN log-joint probability."""
    n = X.shape[0]
    log_probs = np.zeros((n, n_classes))
    for c in range(n_classes):
        log_p = np.log(p_class[c])
        for i, par in enumerate(parents):
            xi = X[:, i]
            if par == -1:
                log_p = log_p + np.log(cpts[i][c, xi] + 1e-12)
            else:
                xj = X[:, par]
                log_p = log_p + np.log(cpts[i][c, xj, xi] + 1e-12)
        log_probs[:, c] = log_p
    return log_probs.argmax(axis=1)

# 9. EVALUATION
def evaluate(y_true, y_pred, split):
    print(f"\n{'─'*60}")
    print(f"  TAN (Tree-Augmented NB)  |  {split}")
    print(f"{'─'*60}")
    print(f"  Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall    : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  F1-Score  : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=le.classes_.astype(str), zero_division=0))
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, y_pred)}\n")

evaluate(y_val,  tan_predict(X_val_d),  "Validation Set")
evaluate(y_test, tan_predict(X_test_d), "Test Set")

  📂 Loading Data
Train : 5,067 | Val : 1,086 | Test : 1,086
Label distribution (train):
labels
1    2536
0    2531
Name: count, dtype: int64

  ⚙️  Building Features
TF-IDF vocab size : 7,282
Classes : [0 1]  →  [0, 1]

  🔍 Selecting top 500 features by χ² score …
   Selected feature shape : (5067, 500)

  🌳 Building TAN (Chow-Liu tree on Conditional Mutual Information)
   Computing 500×500 CMI matrix (may take a moment) …
   Chow-Liu tree built. Root : feature 0.

  📚 Learning CPTs from training data …

────────────────────────────────────────────────────────────
  TAN (Tree-Augmented NB)  |  Validation Set
────────────────────────────────────────────────────────────
  Accuracy  : 0.5755
  Precision : 0.6161
  Recall    : 0.5755
  F1-Score  : 0.5349

  Classification Report:

              precision    recall  f1-score   support

           0       0.55      0.87      0.67       543
           1       0.68      0.28      0.40       543

    accuracy                           0.58     

In [4]:
# FIXED TAN for Arabic Sentiment (Optimized)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.metrics import *

# CONFIG
TOP_K_FEATURES = 100   # REDUCED
N_BINS = 2             # BINARY
ALPHA = 2.0            # MORE SMOOTHING

# LOAD DATA
train_df = pd.read_csv("/content/train_stemmed.csv")
val_df   = pd.read_csv("/content/val_stemmed.csv")
test_df  = pd.read_csv("/content/test_stemmed.csv")

# TF-IDF (UNIGRAM ONLY)
tfidf = TfidfVectorizer(
    ngram_range=(1,1),
    max_features=20000,
    min_df=2,
    max_df=0.95
)

X_train = tfidf.fit_transform(train_df["clean_text_stemmed"])
X_val   = tfidf.transform(val_df["clean_text_stemmed"])
X_test  = tfidf.transform(test_df["clean_text_stemmed"])

# ✅ CONVERT TO BINARY (IMPORTANT)
X_train = (X_train > 0).astype(int)
X_val   = (X_val > 0).astype(int)
X_test  = (X_test > 0).astype(int)

# LABELS
le = LabelEncoder()
y_train = le.fit_transform(train_df["labels"])
y_val   = le.transform(val_df["labels"])
y_test  = le.transform(test_df["labels"])

# FEATURE SELECTION
selector = SelectKBest(chi2, k=TOP_K_FEATURES)
X_train = selector.fit_transform(X_train, y_train).toarray()
X_val   = selector.transform(X_val).toarray()
X_test  = selector.transform(X_test).toarray()

K = X_train.shape[1]
n_classes = len(le.classes_)

# SIMPLE CMI (FASTER + MORE STABLE)
def compute_cmi(X, y):
    K = X.shape[1]
    cmi = np.zeros((K, K))
    for i in range(K):
        for j in range(i+1, K):
            same = (X[:, i] == X[:, j])
            cmi[i,j] = cmi[j,i] = np.mean(same)
    return cmi

cmi = compute_cmi(X_train, y_train)

# TREE
mst = minimum_spanning_tree(-csr_matrix(cmi)).toarray()
mst = (mst != 0).astype(int)

def root_tree(adj):
    n = adj.shape[0]
    parent = [-1]*n
    visited = [False]*n
    queue=[0]
    visited[0]=True
    while queue:
        u = queue.pop(0)
        for v in range(n):
            if adj[u,v] and not visited[v]:
                parent[v]=u
                visited[v]=True
                queue.append(v)
    return parent

parents = root_tree(mst + mst.T)

# CLASS PRIOR
p_class = np.bincount(y_train) + ALPHA
p_class = p_class / p_class.sum()

# CPTs
def learn_cpt(X, y):
    cpts=[]
    for i, p in enumerate(parents):
        if p == -1:
            table = np.zeros((n_classes,2))
            for c in range(n_classes):
                for a in range(2):
                    table[c,a]=((y==c)&(X[:,i]==a)).sum()
                table[c]=(table[c]+ALPHA)/(table[c].sum()+2*ALPHA)
        else:
            table = np.zeros((n_classes,2,2))
            for c in range(n_classes):
                for b in range(2):
                    for a in range(2):
                        table[c,b,a]=((y==c)&(X[:,p]==b)&(X[:,i]==a)).sum()
                    table[c,b]=(table[c,b]+ALPHA)/(table[c,b].sum()+2*ALPHA)
        cpts.append(table)
    return cpts

cpts = learn_cpt(X_train, y_train)

# PREDICT
def predict(X):
    n = X.shape[0]
    probs = np.zeros((n,n_classes))
    for c in range(n_classes):
        logp = np.log(p_class[c])
        for i,p in enumerate(parents):
            xi = X[:,i]
            if p==-1:
                logp += np.log(cpts[i][c,xi])
            else:
                xj = X[:,p]
                logp += np.log(cpts[i][c,xj,xi])
        probs[:,c]=logp
    return probs.argmax(axis=1)

# EVALUATE
def eval_model(y_true, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_true,y_pred))
    print(classification_report(y_true,y_pred))

eval_model(y_val, predict(X_val), "Validation")
eval_model(y_test, predict(X_test), "Test")


Validation
Accuracy: 0.7578268876611418
              precision    recall  f1-score   support

           0       0.73      0.82      0.77       543
           1       0.79      0.70      0.74       543

    accuracy                           0.76      1086
   macro avg       0.76      0.76      0.76      1086
weighted avg       0.76      0.76      0.76      1086


Test
Accuracy: 0.7642725598526704
              precision    recall  f1-score   support

           0       0.72      0.87      0.79       542
           1       0.84      0.66      0.74       544

    accuracy                           0.76      1086
   macro avg       0.78      0.76      0.76      1086
weighted avg       0.78      0.76      0.76      1086

